# MRI ↔ PET Augmentation Pipeline — Combined Output

Copies all **200 originals** and generates **8 augmented variants per patient** into a single new folder `NACC_Combined/`.  
After running, just point `DATA_ROOT` in your training notebook at that one folder — nothing else changes.

```
NACC_Combined/
  NACC000001_orig/          ← original, copied as-is
  NACC000001_aug_0001/      ← augmented variant 1
  NACC000001_aug_0002/
  ...                       (8 variants per patient)
  NACC000001_aug_0008/
  NACC000002_orig/
  ...
  NACC000200_aug_0008/
```

**Total: 200 originals + 1 600 augmented = 1 800 pairs**

| Operation | Applied to | Notes |
|---|---|---|
| Axis flip (L-R, A-P, S-I) | MRI + PET | Random per-axis |
| 90 deg axial rotation | MRI + PET | 0/90/180/270 deg |
| Small continuous rotation | MRI + PET | +/-10 deg in-plane |
| Zoom / scale | MRI + PET | +/-10%, centre-crop back |
| Elastic deformation | MRI + PET | Mild alpha=8, sigma=3 |
| Gamma correction | MRI only | Simulates scanner contrast |
| Contrast jitter | MRI only | Scale around mean |
| Gaussian noise | MRI only | Low sigma <= 0.02 |
| Bias-field simulation | MRI only | Low-order polynomial |
| Intensity shift | MRI only | +/-5% global |

PET is never touched by intensity ops — spatial correspondence is always preserved.

In [ ]:
# Cell 1 : Install dependencies
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "scipy", "scikit-image", "tqdm", "matplotlib"],
    check=False
)

import os, random, time, shutil
import numpy as np
from scipy.ndimage import rotate as scipy_rotate, map_coordinates, gaussian_filter
from skimage.transform import resize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

print('Dependencies ready.')

Dependencies ready.


In [ ]:
import glob
matches = glob.glob('/content/drive/MyDrive/**/mri_processed.npy', recursive=True)
print(matches[:5])

['/content/drive/MyDrive/2/NACC_Test_Preprocessed/NACC000314/mri_processed.npy', '/content/drive/MyDrive/2/NACC_Test_Preprocessed/NACC001978/mri_processed.npy', '/content/drive/MyDrive/2/NACC_Test_Preprocessed/NACC001129/mri_processed.npy', '/content/drive/MyDrive/2/NACC_Test_Preprocessed/NACC001279/mri_processed.npy', '/content/drive/MyDrive/NACC_Test_Preprocessed_Registered2/NACC001279/mri_processed.npy']


In [ ]:
# Cell 2 : Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# ── Diagnostic + DICOM cleanup ─────────────────────────────────
import os, glob

# DATA_ROOT = '/content/drive/MyDrive/NACC_Matched_Pairs_Enhanced_Preprocessed/NACC_Matched_Pairs_Enhanced_Preprocessed'

DATA_ROOT = '/content/drive/MyDrive/NACC_Matched_Pairs_Enhanced_Preprocessed/NACC_Matched_Pairs_Enhanced_Preprocessed'

# Common DICOM extensions and identifiers
DICOM_EXTENSIONS = {'.dcm', '.ima', '.dicom', ''}   # '' = no extension (raw DICOM)
DICOM_PREFIXES   = {'IM_', 'MR.', 'CT.', 'PT.'}

deleted_files  = []
missing_npy    = []

print(f"Scanning: {DATA_ROOT}\n")

for entry in sorted(os.listdir(DATA_ROOT)):
    sd = os.path.join(DATA_ROOT, entry)
    if not os.path.isdir(sd):
        continue

    contents = os.listdir(sd)
    has_mri  = 'mri_processed.npy' in contents
    has_pet  = 'pet_processed.npy' in contents

    print(f"  {entry}/  (mri_npy={has_mri}, pet_npy={has_pet})")
    for f in sorted(contents):
        fp  = os.path.join(sd, f)
        ext = os.path.splitext(f)[1].lower()
        is_dicom = (
            ext in DICOM_EXTENSIONS
            and ext != '.npy'          # never delete .npy
            and ext != '.png'          # keep QC images
            and (
                ext == '.dcm'
                or ext == '.ima'
                or ext == '.dicom'
                or any(f.startswith(p) for p in DICOM_PREFIXES)
                or (ext == '' and f not in ('mri_processed.npy', 'pet_processed.npy'))
            )
        )
        marker = '  [DICOM - will delete]' if is_dicom else ''
        print(f"      {f}{marker}")

    if not (has_mri and has_pet):
        missing_npy.append(entry)

print(f"\n{'='*55}")
print(f"Folders missing .npy files: {len(missing_npy)}")
for m in missing_npy[:10]:
    print(f"  {m}")

Scanning: /content/drive/MyDrive/NACC_Matched_Pairs_Enhanced_Preprocessed/NACC_Matched_Pairs_Enhanced_Preprocessed

  NACC000133/  (mri_npy=True, pet_npy=True)
      NACC000133_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC000314/  (mri_npy=True, pet_npy=True)
      NACC000314_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC000441/  (mri_npy=True, pet_npy=True)
      NACC000441_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC000806/  (mri_npy=True, pet_npy=True)
      NACC000806_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC000868/  (mri_npy=True, pet_npy=True)
      NACC000868_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC001129/  (mri_npy=True, pet_npy=True)
      NACC001129_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC001279/  (mri_npy=True, pet_npy=True)
      NACC001279_qc.png
      mri_processed.npy
      pet_processed.npy
  NACC001978/  (mri_npy=True, pet_npy=True)
      NACC001978_qc.png
    

In [ ]:
# ── DELETE DICOM files (run only after confirming output above) ─
DICOM_EXTENSIONS = {'.dcm', '.ima', '.dicom', ''}
DICOM_PREFIXES   = {'IM_', 'MR.', 'CT.', 'PT.'}

# deleted = []

# for entry in sorted(os.listdir(DATA_ROOT)):
#     sd = os.path.join(DATA_ROOT, entry)
#     if not os.path.isdir(sd):
#         continue
#     for f in os.listdir(sd):
#         fp  = os.path.join(sd, f)
#         ext = os.path.splitext(f)[1].lower()
#         if (
#             ext != '.npy'
#             and ext != '.png'
#             and (
#                 ext in {'.dcm', '.ima', '.dicom'}
#                 or any(f.startswith(p) for p in DICOM_PREFIXES)
#                 or (ext == '' and not f.startswith('.'))
#             )
#         ):
#             os.remove(fp)
#             deleted.append(os.path.join(entry, f))

# print(f"Deleted {len(deleted)} DICOM file(s):")
# for d in deleted[:20]:
#     print(f"  {d}")
# if len(deleted) > 20:
#     print(f"  ... and {len(deleted)-20} more")

deleted = []

for entry in sorted(os.listdir(DATA_ROOT)):
    sd = os.path.join(DATA_ROOT, entry)
    if not os.path.isdir(sd):
        continue

    for f in os.listdir(sd):
        fp = os.path.join(sd, f)

        # 🔴 Skip directories
        if not os.path.isfile(fp):
            continue

        ext = os.path.splitext(f)[1].lower()

        if (
            ext != '.npy'
            and ext != '.png'
            and (
                ext in {'.dcm', '.ima', '.dicom'}
                or any(f.startswith(p) for p in DICOM_PREFIXES)
                or (ext == '' and not f.startswith('.'))
            )
        ):
            os.remove(fp)
            deleted.append(os.path.join(entry, f))

print(f"Deleted {len(deleted)} DICOM file(s):")
for d in deleted[:20]:
    print(f"  {d}")
if len(deleted) > 20:
    print(f"  ... and {len(deleted)-20} more")

Deleted 0 DICOM file(s):


In [ ]:
import os
import shutil

deleted_subjects = []

for entry in sorted(os.listdir(DATA_ROOT)):
    sd = os.path.join(DATA_ROOT, entry)

    if not os.path.isdir(sd):
        continue

    contents = os.listdir(sd)
    has_mri = 'mri_processed.npy' in contents
    has_pet = 'pet_processed.npy' in contents

    # ❌ If missing either → delete whole folder
    if not (has_mri and has_pet):
        shutil.rmtree(sd)
        deleted_subjects.append(entry)

print(f"Deleted {len(deleted_subjects)} subject folders:")
for d in deleted_subjects[:20]:
    print(f"  {d}")
if len(deleted_subjects) > 20:
    print(f"  ... and {len(deleted_subjects)-20} more")

Deleted 45 subject folders:
  NACC114717
  NACC119068
  NACC139587
  NACC143450 (1)
  NACC145236 (1)
  NACC146294 (1)
  NACC147135 (1)
  NACC147464 (1)
  NACC148143 (1)
  NACC149999 (1)
  NACC150150 (1)
  NACC150247 (1)
  NACC155205 (1)
  NACC161404 (1)
  NACC161833 (1)
  NACC164649 (1)
  NACC165089 (1)
  NACC169557 (1)
  NACC170509 (1)
  NACC172611 (1)
  ... and 25 more


In [ ]:
# Cell 3 : Configuration
# ----------------------------------------------------------------
# UPDATE THESE TWO PATHS TO MATCH YOUR GOOGLE DRIVE LAYOUT
# ----------------------------------------------------------------

# Folder containing your 200 original patient folders
DATA_ROOT = '/content/drive/MyDrive/NACC_Matched_Pairs_Enhanced_Preprocessed/NACC_Matched_Pairs_Enhanced_Preprocessed'

# New combined output folder (created automatically)
# Originals AND augmented both go here.
OUT_ROOT  = '/content/drive/MyDrive/NACC_Combined_AUG'

# 200 patients x 8 = 1 600 augmented  +  200 originals = 1 800 total
N_AUG_PER_SUBJECT = 8

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# # Auto-discover source subjects
# SUBJECTS = []
# # ── Diagnostic: run this before the vol_bytes line ──
# print(f"DATA_ROOT exists: {os.path.exists(DATA_ROOT)}")
# print(f"\nFirst 10 entries in DATA_ROOT:")
# for entry in sorted(os.listdir(DATA_ROOT))[:10]:
#     sd  = os.path.join(DATA_ROOT, entry)
#     mri = os.path.join(sd, 'mri_processed.npy')
#     pet = os.path.join(sd, 'pet_processed.npy')
#     print(f"  {entry}/")
#     print(f"    is_dir={os.path.isdir(sd)}  mri={os.path.isfile(mri)}  pet={os.path.isfile(pet)}")

# vol_bytes  = os.path.getsize(SUBJECTS[0]['mri'])
# total_orig = len(SUBJECTS)
# total_aug  = len(SUBJECTS) * N_AUG_PER_SUBJECT
# total_all  = total_orig + total_aug
# est_gb     = (total_all * 2 * vol_bytes) / 1e9

# print(f'Found {total_orig} source subjects')
# print(f'Plan  : {total_orig} originals + {total_aug} augmented = {total_all} total pairs')
# print(f'Output: {OUT_ROOT}  (~{est_gb:.1f} GB)')


# Build SUBJECTS list
SUBJECTS = []

for entry in sorted(os.listdir(DATA_ROOT)):
    sd = os.path.join(DATA_ROOT, entry)

    if not os.path.isdir(sd):
        continue

    mri = os.path.join(sd, 'mri_processed.npy')
    pet = os.path.join(sd, 'pet_processed.npy')

    if os.path.isfile(mri) and os.path.isfile(pet):
        SUBJECTS.append({
            'id': entry,
            'mri': mri,
            'pet': pet
        })

# Now safe to use
vol_bytes  = os.path.getsize(SUBJECTS[0]['mri'])
total_orig = len(SUBJECTS)
total_aug  = len(SUBJECTS) * N_AUG_PER_SUBJECT
total_all  = total_orig + total_aug
est_gb     = (total_all * 2 * vol_bytes) / 1e9

print(f'Found {total_orig} source subjects')
print(f'Plan  : {total_orig} originals + {total_aug} augmented = {total_all} total pairs')
print(f'Output: {OUT_ROOT}  (~{est_gb:.1f} GB)')


Found 259 source subjects
Plan  : 259 originals + 2072 augmented = 2331 total pairs
Output: /content/drive/MyDrive/NACC_Combined_AUG  (~91.7 GB)


In [ ]:
# Cell 4 : Normalisation
def norm(v):
    lo, hi = np.percentile(v, 1), np.percentile(v, 99)
    return np.clip((v - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

In [ ]:
# Cell 5 : Augmentation suite
# Spatial ops  -> applied identically to MRI + PET
# Intensity ops -> MRI only

def aug_flip(mri, pet):
    for ax in range(3):
        if random.random() > 0.5:
            mri = np.flip(mri, axis=ax).copy()
            pet = np.flip(pet, axis=ax).copy()
    return mri, pet

def aug_rot90(mri, pet):
    k = random.randint(0, 3)
    if k:
        mri = np.rot90(mri, k=k, axes=(1, 2)).copy()
        pet = np.rot90(pet, k=k, axes=(1, 2)).copy()
    return mri, pet

def aug_rotate_small(mri, pet, max_angle=10):
    angle = random.uniform(-max_angle, max_angle)
    mri = scipy_rotate(mri, angle, axes=(1, 2), reshape=False, order=1, mode='nearest')
    pet = scipy_rotate(pet, angle, axes=(1, 2), reshape=False, order=1, mode='nearest')
    return mri.astype(np.float32), pet.astype(np.float32)

def aug_zoom(mri, pet, lo=0.9, hi=1.1):
    factor = random.uniform(lo, hi)
    D, H, W = mri.shape
    nD, nH, nW = int(D*factor), int(H*factor), int(W*factor)
    def _r(v):
        return resize(v, (nD, nH, nW), order=1, mode='reflect',
                      anti_aliasing=False, preserve_range=True).astype(np.float32)
    def _cp(v, tD, tH, tW):
        out = np.zeros((tD, tH, tW), dtype=np.float32)
        sd=min(v.shape[0],tD); sh=min(v.shape[1],tH); sw=min(v.shape[2],tW)
        od=(v.shape[0]-sd)//2; oh=(v.shape[1]-sh)//2; ow=(v.shape[2]-sw)//2
        td=(tD-sd)//2; th=(tH-sh)//2; tw=(tW-sw)//2
        out[td:td+sd, th:th+sh, tw:tw+sw] = v[od:od+sd, oh:oh+sh, ow:ow+sw]
        return out
    return _cp(_r(mri),D,H,W), _cp(_r(pet),D,H,W)

def aug_elastic(mri, pet, alpha=8, sigma=3):
    shape = mri.shape
    dx = gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma) * alpha
    dy = gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma) * alpha
    dz = gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma) * alpha
    x,y,z = np.meshgrid(np.arange(shape[0]),np.arange(shape[1]),np.arange(shape[2]),indexing='ij')
    coords = [np.clip(x+dx,0,shape[0]-1), np.clip(y+dy,0,shape[1]-1), np.clip(z+dz,0,shape[2]-1)]
    return (map_coordinates(mri,coords,order=1,mode='nearest').astype(np.float32),
            map_coordinates(pet,coords,order=1,mode='nearest').astype(np.float32))

def aug_gamma(mri):
    return np.power(np.clip(mri, 1e-6, 1.0), random.uniform(0.7, 1.5)).astype(np.float32)

def aug_contrast_jitter(mri):
    m = mri.mean()
    return np.clip((mri-m)*random.uniform(0.8,1.2)+m, 0.0, 1.0).astype(np.float32)

def aug_gaussian_noise(mri):
    return np.clip(mri + np.random.randn(*mri.shape).astype(np.float32)*random.uniform(0,0.02), 0.0, 1.0)

def aug_bias_field(mri, order=3):
    D,H,W = mri.shape
    coords = np.mgrid[0:D, 0:H, 0:W].astype(np.float32)
    for i,s in enumerate([D,H,W]): coords[i] = coords[i]/(s-1)*2-1
    field = np.ones((D,H,W), dtype=np.float32)
    for _ in range(order): field += random.uniform(-0.1,0.1)*coords[random.randint(0,2)]
    return np.clip(mri*np.clip(field,0.8,1.2), 0.0, 1.0).astype(np.float32)

def aug_intensity_shift(mri):
    return np.clip(mri+random.uniform(-0.05,0.05), 0.0, 1.0).astype(np.float32)

def augment_pair(mri, pet):
    mri, pet = aug_flip(mri, pet)
    mri, pet = aug_rot90(mri, pet)
    if random.random() < 0.6:  mri, pet = aug_rotate_small(mri, pet)
    if random.random() < 0.4:  mri, pet = aug_zoom(mri, pet)
    if random.random() < 0.3:  mri, pet = aug_elastic(mri, pet)
    if random.random() < 0.5:  mri = aug_gamma(mri)
    if random.random() < 0.5:  mri = aug_contrast_jitter(mri)
    if random.random() < 0.4:  mri = aug_gaussian_noise(mri)
    if random.random() < 0.35: mri = aug_bias_field(mri)
    if random.random() < 0.4:  mri = aug_intensity_shift(mri)
    return mri.copy(), pet.copy()

print('Augmentation suite ready.')

Augmentation suite ready.


In [ ]:
# Cell 6 : Sanity-check preview (one subject, one augmentation)
s0       = SUBJECTS[0]
mri_orig = norm(np.load(s0['mri']))
pet_orig = norm(np.load(s0['pet']))
print(f"Preview: {s0['id']}  shape={mri_orig.shape}")

random.seed(7); np.random.seed(7)
mri_aug, pet_aug = augment_pair(mri_orig.copy(), pet_orig.copy())

mid = mri_orig.shape[0] // 2
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panels = [
    (mri_orig[mid],                         'Original MRI axial',    'gray'),
    (mri_orig[:, mri_orig.shape[1]//2, :],  'Original MRI coronal',  'gray'),
    (mri_orig[:, :, mri_orig.shape[2]//2],  'Original MRI sagittal', 'gray'),
    (mri_aug[mid],                          'Augmented MRI axial',   'gray'),
    (mri_aug[:, mri_aug.shape[1]//2, :],    'Augmented MRI coronal', 'gray'),
    (mri_aug[:, :, mri_aug.shape[2]//2],    'Augmented MRI sagittal','gray'),
]
for ax, (img, title, cmap) in zip(axes.flat, panels):
    ax.imshow(img, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis('off')
plt.suptitle(f'Augmentation preview - {s0["id"]}', fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/aug_preview.png', dpi=120, bbox_inches='tight')
plt.show()
del mri_orig, pet_orig, mri_aug, pet_aug
print('Preview OK. Run Cell 7 to start the full pipeline.')

Preview: NACC000133  shape=(160, 192, 160)
Preview OK. Run Cell 7 to start the full pipeline.


In [ ]:
# Cell 7 : Main pipeline - originals + augmented -> OUT_ROOT
#
# For every subject:
#   Step A) shutil.copy2 original files -> OUT_ROOT/{sid}_orig/
#   Step B) generate N_AUG_PER_SUBJECT augmented variants
#           -> OUT_ROOT/{sid}_aug_0001/ ... OUT_ROOT/{sid}_aug_000N/
#
# All folders contain exactly:  mri_processed.npy + pet_processed.npy
# so DATA_ROOT = OUT_ROOT is the ONLY change needed in the training notebook.

os.makedirs(OUT_ROOT, exist_ok=True)

orig_written = 0
aug_written  = 0
t_start      = time.time()

for subj in SUBJECTS:
    sid = subj['id']
    print(f'\n-- {sid} ----------------------------------------')

    # --- Step A: copy original -----------------------------------
    orig_folder = os.path.join(OUT_ROOT, f'{sid}_orig')
    os.makedirs(orig_folder, exist_ok=True)
    shutil.copy2(subj['mri'], os.path.join(orig_folder, 'mri_processed.npy'))
    shutil.copy2(subj['pet'], os.path.join(orig_folder, 'pet_processed.npy'))
    orig_written += 1
    print(f'   [A] original  -> {sid}_orig/')

    # --- Step B: generate augmented variants ---------------------
    mri_src = norm(np.load(subj['mri']))
    pet_src = norm(np.load(subj['pet']))
    seed_offset = abs(hash(sid)) % (2**16)

    pbar = tqdm(total=N_AUG_PER_SUBJECT, desc='   [B] augmenting', unit='pair', leave=False)
    for i in range(N_AUG_PER_SUBJECT):
        random.seed(GLOBAL_SEED + seed_offset + i)
        np.random.seed(GLOBAL_SEED + seed_offset + i)
        mri_aug, pet_aug = augment_pair(mri_src.copy(), pet_src.copy())
        aug_folder = os.path.join(OUT_ROOT, f'{sid}_aug_{i+1:04d}')
        os.makedirs(aug_folder, exist_ok=True)
        np.save(os.path.join(aug_folder, 'mri_processed.npy'), mri_aug)
        np.save(os.path.join(aug_folder, 'pet_processed.npy'), pet_aug)
        aug_written += 1
        pbar.update(1)
    pbar.close()

    done  = orig_written + aug_written
    total = len(SUBJECTS) * (1 + N_AUG_PER_SUBJECT)
    elapsed = time.time() - t_start
    eta = (elapsed / done) * (total - done) / 60 if done else 0
    print(f'   [B] {N_AUG_PER_SUBJECT} augmented variants saved.  '
          f'Overall: {done}/{total} folders  ETA: {eta:.1f} min')
    del mri_src, pet_src

elapsed_total = time.time() - t_start
print(f"\n{'='*52}")
print(f'  DONE')
print(f'  Originals copied : {orig_written}')
print(f'  Augmented pairs  : {aug_written}')
print(f'  Total folders    : {orig_written + aug_written}')
print(f'  Wall time        : {elapsed_total/60:.1f} min')
print(f'  Output           : {OUT_ROOT}')
print(f"{'='*52}")


-- NACC000133 ----------------------------------------
   [A] original  -> NACC000133_orig/


   [B] augmenting:   0%|          | 0/8 [00:00<?, ?pair/s]

   [B] 8 augmented variants saved.  Overall: 9/2331 folders  ETA: 94.1 min

-- NACC000314 ----------------------------------------
   [A] original  -> NACC000314_orig/


   [B] augmenting:   0%|          | 0/8 [00:00<?, ?pair/s]

   [B] 8 augmented variants saved.  Overall: 18/2331 folders  ETA: 121.2 min

-- NACC000441 ----------------------------------------
   [A] original  -> NACC000441_orig/


...

   [B] 8 augmented variants saved.  Overall: 2313/2331 folders  ETA: 0.7 min

-- NACC213181 ----------------------------------------
   [A] original  -> NACC213181_orig/


   [B] augmenting:   0%|          | 0/8 [00:00<?, ?pair/s]

   [B] 8 augmented variants saved.  Overall: 2322/2331 folders  ETA: 0.4 min

-- NACC213239 ----------------------------------------
   [A] original  -> NACC213239_orig/


   [B] augmenting:   0%|          | 0/8 [00:00<?, ?pair/s]

   [B] 8 augmented variants saved.  Overall: 2331/2331 folders  ETA: 0.0 min

  DONE
  Originals copied : 259
  Augmented pairs  : 2072
  Total folders    : 2331
  Wall time        : 91.6 min
  Output           : /content/drive/MyDrive/NACC_Combined_AUG


In [ ]:
# Cell 8 : Verification
all_folders  = sorted(d for d in os.listdir(OUT_ROOT)
                      if os.path.isdir(os.path.join(OUT_ROOT, d)))
orig_folders = [d for d in all_folders if d.endswith('_orig')]
aug_folders  = [d for d in all_folders if '_aug_' in d]

print(f'Total folders : {len(all_folders)}')
print(f'  _orig       : {len(orig_folders)}')
print(f'  _aug_XXXX   : {len(aug_folders)}')

missing = [d for d in all_folders
           if not (os.path.isfile(os.path.join(OUT_ROOT,d,'mri_processed.npy')) and
                   os.path.isfile(os.path.join(OUT_ROOT,d,'pet_processed.npy')))]
if missing:
    print(f'WARNING: {len(missing)} folders missing files: {missing[:5]}')
else:
    print('OK - every folder has mri_processed.npy + pet_processed.npy')

print('\nSpot-check (3 random folders):')
for d in random.sample(all_folders, min(3, len(all_folders))):
    fd = os.path.join(OUT_ROOT, d)
    m  = np.load(os.path.join(fd, 'mri_processed.npy'))
    p  = np.load(os.path.join(fd, 'pet_processed.npy'))
    tag = '[orig]' if d.endswith('_orig') else '[aug] '
    print(f'  {tag} {d}')
    print(f'         MRI shape={m.shape} min={m.min():.3f} max={m.max():.3f} '
          f'NaN={np.isnan(m).any()} Inf={np.isinf(m).any()}')
    print(f'         PET shape={p.shape} min={p.min():.3f} max={p.max():.3f} '
          f'NaN={np.isnan(p).any()} Inf={np.isinf(p).any()}')

total_bytes = sum(
    os.path.getsize(os.path.join(OUT_ROOT, d, f))
    for d in all_folders
    for f in ['mri_processed.npy', 'pet_processed.npy']
    if os.path.isfile(os.path.join(OUT_ROOT, d, f))
)
print(f'\nTotal storage: {total_bytes/1e9:.2f} GB')

Total folders : 2331
  _orig       : 259
  _aug_XXXX   : 2072
OK - every folder has mri_processed.npy + pet_processed.npy

Spot-check (3 random folders):
  [aug]  NACC020387_aug_0007
         MRI shape=(160, 192, 160) min=0.024 max=0.879 NaN=False Inf=False
         PET shape=(160, 192, 160) min=0.000 max=1.000 NaN=False Inf=False
  [aug]  NACC079008_aug_0007
         MRI shape=(160, 160, 192) min=0.000 max=1.000 NaN=False Inf=False
         PET shape=(160, 160, 192) min=0.000 max=1.000 NaN=False Inf=False
  [aug]  NACC006507_aug_0004
         MRI shape=(160, 192, 160) min=0.000 max=0.893 NaN=False Inf=False
         PET shape=(160, 192, 160) min=0.000 max=1.000 NaN=False Inf=False

Total storage: 91.66 GB


In [ ]:
# Cell 9 : Visual QC grid (9 random folders, mix of orig + aug)
sample_dirs = random.sample(all_folders, min(9, len(all_folders)))
fig, axes   = plt.subplots(3, 3, figsize=(12, 12))
for ax, d in zip(axes.flat, sample_dirs):
    m   = np.load(os.path.join(OUT_ROOT, d, 'mri_processed.npy'))
    tag = 'O' if d.endswith('_orig') else 'A'
    ax.imshow(m[m.shape[0]//2], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'[{tag}] {d}', fontsize=5)
    ax.axis('off')
plt.suptitle('MRI axial mid-slice  O=original  A=augmented', fontsize=10)
plt.tight_layout()
qc_path = os.path.join(OUT_ROOT, 'qc_grid.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'QC grid -> {qc_path}')

QC grid -> /content/drive/MyDrive/NACC_Combined_AUG/qc_grid.png


In [ ]:
# Cell 10 : Intensity histogram - original vs augmented (up to 5 subjects)
sample_subjs = SUBJECTS[:min(5, len(SUBJECTS))]
fig, axes    = plt.subplots(1, len(sample_subjs), figsize=(6*len(sample_subjs), 4))
if len(sample_subjs) == 1: axes = [axes]

for ax, subj in zip(axes, sample_subjs):
    sid      = subj['id']
    orig_dir = os.path.join(OUT_ROOT, f'{sid}_orig')
    mri_o    = np.load(os.path.join(orig_dir, 'mri_processed.npy')).ravel()
    s_aug    = [d for d in aug_folders if d.startswith(sid)]
    aug_data = np.concatenate([
        np.load(os.path.join(OUT_ROOT, d, 'mri_processed.npy')).ravel()
        for d in random.sample(s_aug, min(8, len(s_aug)))
    ])
    ax.hist(mri_o,    bins=80, alpha=0.6, density=True, label='Original',       color='steelblue')
    ax.hist(aug_data, bins=80, alpha=0.6, density=True, label='Augmented (x8)', color='darkorange')
    ax.set_title(sid, fontsize=8)
    ax.set_xlabel('Normalised intensity')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('MRI intensity: original vs augmented', fontsize=11)
plt.tight_layout()
hist_path = os.path.join(OUT_ROOT, 'intensity_histogram.png')
plt.savefig(hist_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Histogram -> {hist_path}')

Histogram -> /content/drive/MyDrive/NACC_Combined_AUG/intensity_histogram.png


In [ ]:
# Cell 11 : One-line change to start training
print(f'''
============================================================
  DONE - make ONE change in simple_mri2pet_hptuner.ipynb
============================================================

  In the Config cell, change DATA_ROOT to:

    DATA_ROOT = "{OUT_ROOT}"

  That is the ONLY change needed.

  The pair-discovery loop already scans every subfolder
  for mri_processed.npy + pet_processed.npy, so it will
  automatically pick up all {len(orig_folders)} _orig and {len(aug_folders)} _aug folders.

  Total pairs available for training: {len(all_folders)}
    {len(orig_folders)} originals  +  {len(aug_folders)} augmented

============================================================
''')


  DONE - make ONE change in simple_mri2pet_hptuner.ipynb

  In the Config cell, change DATA_ROOT to:

    DATA_ROOT = "/content/drive/MyDrive/NACC_Combined_AUG"

  That is the ONLY change needed.

  The pair-discovery loop already scans every subfolder
  for mri_processed.npy + pet_processed.npy, so it will
  automatically pick up all 259 _orig and 2072 _aug folders.

  Total pairs available for training: 2331
    259 originals  +  2072 augmented




In [ ]:
# Cell FIX : Standardise all volumes to TARGET_SHAPE
# -----------------------------------------------------------
# Run ONCE after your existing Cell 8 verification.
# Overwrites any npy whose shape doesn't match TARGET_SHAPE.
# -----------------------------------------------------------

from skimage.transform import resize as sk_resize
import numpy as np, os
from tqdm.notebook import tqdm

# ── Choose target shape ──────────────────────────────────────
# (160, 192, 160) is the majority shape in your dataset.
# Pick whatever is most common, or a power-of-2-friendly cube.
TARGET_SHAPE = (160, 192, 160)

# ── Scan & fix ───────────────────────────────────────────────
all_folders = sorted(
    d for d in os.listdir(OUT_ROOT)
    if os.path.isdir(os.path.join(OUT_ROOT, d))
)

resized_count  = 0
already_ok     = 0
errors         = []

for d in tqdm(all_folders, desc='Checking shapes'):
    fd = os.path.join(OUT_ROOT, d)
    for fname in ('mri_processed.npy', 'pet_processed.npy'):
        fp = os.path.join(fd, fname)
        try:
            vol = np.load(fp)
            if vol.shape == TARGET_SHAPE:
                already_ok += 1
                continue

            # Resize to target (order=1 = trilinear, fast & safe)
            vol_resized = sk_resize(
                vol,
                TARGET_SHAPE,
                order=1,
                mode='reflect',
                anti_aliasing=False,
                preserve_range=True
            ).astype(np.float32)

            np.save(fp, vol_resized)
            resized_count += 1

        except Exception as e:
            errors.append((fp, str(e)))

print(f"\n{'='*50}")
print(f"  Already correct shape : {already_ok}")
print(f"  Resized to {TARGET_SHAPE}  : {resized_count}")
print(f"  Errors                : {len(errors)}")
for fp, msg in errors[:5]:
    print(f"    {fp}  →  {msg}")
print(f"{'='*50}")

Checking shapes:   0%|          | 0/2331 [00:00<?, ?it/s]


  Already correct shape : 2584
  Resized to (160, 192, 160)  : 2078
  Errors                : 0


In [ ]:
# Cell FIX-VERIFY : Confirm all shapes are now uniform
shape_counts = {}

for d in sorted(os.listdir(OUT_ROOT)):
    fd = os.path.join(OUT_ROOT, d)
    if not os.path.isdir(fd):
        continue
    fp = os.path.join(fd, 'mri_processed.npy')
    if os.path.isfile(fp):
        s = np.load(fp, mmap_mode='r').shape
        shape_counts[s] = shape_counts.get(s, 0) + 1

print("Shape distribution across all folders:")
for shape, count in sorted(shape_counts.items(), key=lambda x: -x[1]):
    flag = "✓" if shape == TARGET_SHAPE else "✗ MISMATCH"
    print(f"  {shape}  →  {count:4d} folders  {flag}")

Shape distribution across all folders:
  (160, 192, 160)  →  2331 folders  ✓


# ZIP

In [ ]:
from google.colab import drive
import os
import shutil
from datetime import datetime

drive.mount('/content/drive')

INPUT_DIR = '/content/drive/MyDrive/NACC_Combined_AUG'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_NAME = f'/content/drive/MyDrive/NACC_Combined_AUG_ZIP_{timestamp}.zip'

if not os.path.exists(INPUT_DIR):
    print("❌ Folder not found:", INPUT_DIR)
    !ls -la '/content/drive/MyDrive/'  # check what's there
else:
    # Unique name prevents overwrite/Trash
    shutil.make_archive(
        base_name=ZIP_NAME[:-4],  # remove .zip extension
        format='zip',
        root_dir=os.path.dirname(INPUT_DIR),
        base_dir=os.path.basename(INPUT_DIR)
    )
    print(f"✅ Zipped to: {ZIP_NAME}")
    # Verify size
    !ls -lh "{ZIP_NAME}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Zipped to: /content/drive/MyDrive/NACC_Combined_AUG_ZIP_20260326_012614.zip
-rw------- 1 root root 46G Mar 26 04:35 /content/drive/MyDrive/NACC_Combined_AUG_ZIP_20260326_012614.zip


In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
auth.authenticate_user()
drive_service = build('drive', 'v3')

results = drive_service.files().list(
    q="name contains 'NACC' and trashed=false",
    fields="files(id, name, parents, mimeType)"
).execute()
items = results.get('files', [])
for item in items:
    print(f"📁 {item['name']} (ID: {item['id']})")


📁 NACC213239_aug_0008 (ID: 1uukRiNM6BP2-UYFERQg0fU4R2rcn5ADj)
📁 NACC213239_aug_0007 (ID: 1hLdJ-NQGgqH60Bw18y60k8gVoWOnDbdW)
📁 NACC213239_aug_0006 (ID: 1dWMEhG1LKjmcp2hKH-8oq1TmMnAeec7f)
📁 NACC213239_aug_0005 (ID: 1kxNwx_m4BLHpkUwhLZaDnV9Hyylo9Bwz)
📁 NACC213239_aug_0004 (ID: 192pq61PxB64xeMHg6_lhmNJayTxj3m8_)
📁 NACC213239_aug_0003 (ID: 1NPOu_ruSN3hfFBbWLAA8UlZ-N12HZIFU)
📁 NACC213239_aug_0002 (ID: 1LiJzxqbALuX8VTlZJRVZnOguhEZtDtZ3)
📁 NACC213239_aug_0001 (ID: 10XqpcATnraT75CBr2Hd4EES74gB21IgZ)
📁 NACC213239_orig (ID: 1UjVbjUEDDsjfgaeQRJVxWuXDd0eFJbWI)
📁 NACC213181_aug_0008 (ID: 11m7au8tDsCziQiW4lil_xtQnwZem97p_)
📁 NACC213181_aug_0007 (ID: 19GoF6EeSNGh37jbaI3w75gIbxRQEhHm8)
📁 NACC213181_aug_0006 (ID: 1-Wkn3bUvFWQgRZQ1nBlAyPMJ7ua9ZFzA)
📁 NACC213181_aug_0005 (ID: 11KnAJd6wVuv4SHXhviGH8qHUC83nK96a)
📁 NACC213181_aug_0004 (ID: 1nxklfYQzv84-mjB9rNGCbeLRApF6hzuR)
📁 NACC213181_aug_0003 (ID: 1MDIGFmytIqQdMN5eKvQdpnJ-57dqh_LY)
📁 NACC213181_aug_0002 (ID: 1yKe5n1nf3mvmKP_JGPb1orymLv5t9m4r)
📁 NACC213181